# Nestlé HR Policy Assistant — Retrieval-Augmented Generation (RAG)

**Course-end assessment | Python + LangChain + ChromaDB + OpenRouter + Gradio**

**Objective.** Build a chatbot that answers questions grounded in *The Nestlé Human Resources Policy (September 2012)*, with retrieved PDF-page references and a user-friendly Gradio interface.

**Architecture:** PDF → PyPDFLoader → document cleaning → text chunks → Hugging Face embeddings → ChromaDB → similarity retrieval → prompt template → OpenRouter LLM → Gradio.

**Assessment adaptation:** The assessment mentions OpenAI embeddings and GPT-3.5 Turbo. This implementation instead uses a local Hugging Face embedding model and an LLM through an OpenAI-compatible **OpenRouter** endpoint, because an OpenRouter key is available. The RAG principles and workflow remain the same. The model selector `openrouter/free` can route to different underlying models, so responses may vary.

## 1. Install dependencies

Install the libraries needed by this notebook. In Google Colab, `%pip` targets the active notebook environment. Dependency warnings involving unrelated preinstalled Colab packages do not necessarily mean this workflow is broken.

In [ ]:
%pip install -q langchain langchain-community langchain-text-splitters langchain-chroma langchain-huggingface langchain-openai sentence-transformers chromadb pypdf gradio

## 2. Load the Nestlé HR Policy PDF

`PyPDFLoader` converts PDF pages into LangChain `Document` objects containing `page_content` and `metadata`. Upload the provided PDF to Colab if it is not already in `/content`. Colab file storage is temporary.

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

pdf_path = Path("/content/the_nestle_hr_policy_pdf_2012.pdf")
if not pdf_path.exists():
    from google.colab import files
    print("Upload the Nestlé HR Policy PDF now.")
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("No PDF uploaded.")
    pdf_path = Path("/content") / next(iter(uploaded))

loader = PyPDFLoader(str(pdf_path))
documents = loader.load()
print("PDF pages loaded:", len(documents))

## 3. Clean the documents

Exclude pages without extracted text and the title-only cover page (PDF page 1). We preserve each page's original metadata so source-page numbers remain correct.

In [ ]:
policy_documents = [
    doc for doc in documents
    if doc.page_content.strip() and doc.metadata.get("page") != 0
]
print("Content pages retained:", len(policy_documents))

## 4. Chunk the extracted text

`RecursiveCharacterTextSplitter` produces smaller text passages suitable for semantic retrieval. `chunk_size=800` and `chunk_overlap=150` are **character** settings. The overlap reduces, but does not eliminate, context loss at chunk boundaries.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)
policy_chunks = text_splitter.split_documents(policy_documents)
print("Policy chunks:", len(policy_chunks))
print("Example source page:", policy_chunks[0].metadata["page"] + 1)
print("Example chunk preview:", policy_chunks[0].page_content[:180])

## 5. Load an embedding model

`sentence-transformers/all-MiniLM-L6-v2` maps text into **384-dimensional** vectors. We use the same model to embed document chunks and questions; this does not require an OpenAI or OpenRouter API key.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
example_vector = embedding_model.embed_query("employee training")
print("Embedding dimensions:", len(example_vector))

## 6. Create the ChromaDB index

ChromaDB stores chunks, embeddings, and page metadata. Stable document IDs allow repeated execution to **upsert** the same chunks rather than append duplicates. We do **not** delete an open SQLite/Chroma directory, which previously caused a read-only database error in the exploratory notebook.

For a new Colab runtime, the index will be recreated in `/content`. For later reruns in the same runtime, identical IDs prevent duplicate records.

In [ ]:
import hashlib
from langchain_chroma import Chroma

chunk_ids = [
    hashlib.sha256(
        f"{i}|{doc.metadata.get('page')}|{doc.page_content}".encode("utf-8")
    ).hexdigest()
    for i, doc in enumerate(policy_chunks)
]

vector_store = Chroma.from_documents(
    documents=policy_chunks,
    embedding=embedding_model,
    ids=chunk_ids,
    collection_name="nestle_hr_policy_assessment",
    persist_directory="/content/nestle_hr_chroma_assessment"
)
print("Indexed chunks:", vector_store._collection.count())

## 7. Demonstrate semantic retrieval

A question is embedded with the same model and matched to the closest document vectors. Returned pages are **retrieved pages**, not verified claim-by-claim citations. Similarity ranking may occasionally favor broad or unrelated content.

In [ ]:
query = "What does Nestlé say about employee training and development?"
retrieved_docs = vector_store.similarity_search(query, k=3)

for rank, doc in enumerate(retrieved_docs, 1):
    print(f"Result {rank} — PDF page {doc.metadata['page'] + 1}")
    print(doc.page_content[:250], "\n")

## 8. Configure the OpenRouter LLM securely

Enter the OpenRouter API key using a hidden input. **Do not paste a key into notebook source code or save its value in outputs.** `ChatOpenAI` works with OpenRouter's OpenAI-compatible API when `base_url` is set. The `/free` router may choose different models and has usage/availability limitations.

The earlier experiments showed that a 100-token limit could be exhausted by model reasoning, producing an empty visible answer. We use `max_tokens=1000` to allow more output.

In [ ]:
from getpass import getpass
from langchain_openai import ChatOpenAI

openrouter_api_key = getpass("Enter your OpenRouter API key: ")
if not openrouter_api_key.strip():
    raise ValueError("OpenRouter API key is required.")

llm = ChatOpenAI(
    model="openrouter/free",
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
    max_tokens=1000,
    request_timeout=60,
    max_retries=0,
    use_responses_api=False
)
print("OpenRouter client configured.")

## 9. Define the grounded prompt

The system prompt limits the answer to supplied policy passages and specifies a fallback when the passages do not contain an answer. These instructions **reduce** hallucination risk but cannot guarantee perfect grounding.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful Nestlé HR Policy Assistant.
Answer questions using only the provided HR policy context.
Do not invent policies, benefits, numerical entitlements, or procedures.
If the context does not contain the answer, respond exactly:
I could not find this information in the provided Nestlé HR Policy.
Be accurate, concise and professional.

HR Policy Context:
{context}"""),
    ("human", "{question}")
])
print("Prompt template ready.")

## 10. Build the RAG question-answering function

The function embeds/searches the query, inserts retrieved chunks into the prompt, calls OpenRouter, and returns the answer with the **retrieved** PDF-page numbers. A small, transparent query expansion improves retrieval for performance-management questions; this is a targeted rule, not a general query-rewriting system.

In [ ]:
def rag_answer_with_sources(question):
    search_query = question
    if "performance" in question.lower():
        search_query += " evaluation objectives feedback coaching"

    docs = vector_store.similarity_search(search_query, k=3)
    context = "\n\n".join(doc.page_content for doc in docs)
    prompt = prompt_template.invoke({"context": context, "question": question})
    response = llm.invoke(prompt)
    answer = response.content if isinstance(response.content, str) else str(response.content)
    pages = sorted({doc.metadata["page"] + 1 for doc in docs})
    return answer, pages

print("RAG function ready.")

## 11. Evaluate with in-scope and out-of-scope questions

Test that a policy question produces a grounded response and that a missing entitlement is **not fabricated**. A generated answer may be incomplete if retrieval misses a relevant passage. Check retrieved pages and compare claims with the PDF.

**Observed during development:** Training and performance questions produced policy-grounded answers; the unsupported question about exact annual leave days correctly returned the missing-information fallback. These observations are not substitutes for rerunning the clean notebook.

In [ ]:
test_questions = [
    "What is Nestlé's policy on workplace harassment and discrimination?",
    "How does Nestlé handle employee performance?",
    "How many days of annual paid leave does Nestlé provide?"
]

for question in test_questions:
    answer, pages = rag_answer_with_sources(question)
    print("QUESTION:", question)
    print("ANSWER:", answer or "[Empty model response]")
    if answer and not answer.strip().startswith("I could not find this information"):
        print("Retrieved PDF pages:", pages)
    print("-" * 70)

## 12. Launch the Gradio chatbot

`gr.ChatInterface` creates a conversational UI connected to the RAG function. The `history` parameter is accepted for the interface but **not yet used** to resolve follow-up questions; each question is retrieved independently.

In Colab, Gradio may create a temporary public `gradio.live` link. **Anyone with that link may submit questions using your OpenRouter API quota.** Stop the notebook cell when the demo is finished; do not present the link as permanent hosting.

In [ ]:
import gradio as gr

def chatbot_response(message, history):
    try:
        answer, pages = rag_answer_with_sources(message)
        if not answer or not answer.strip():
            return "I couldn't generate a response. Please try again."
        if answer.strip().startswith("I could not find this information"):
            return answer
        return answer + "\n\n**Retrieved PDF pages:** " + ", ".join(map(str, pages))
    except Exception:
        return "The model request failed. Please try again later."

demo = gr.ChatInterface(
    fn=chatbot_response,
    title="Nestlé HR Policy Assistant",
    description="Ask questions about the Nestlé Human Resources Policy (2012).",
    examples=[
        "What is Nestlé's policy on employee training?",
        "How does Nestlé handle employee performance?",
        "What is Nestlé's policy on workplace harassment and discrimination?",
        "How many days of annual paid leave does Nestlé provide?"
    ]
)

demo.launch(share=True, debug=True)

## 13. Results, limitations and conclusion

**Result.** The chatbot implements the required document ingestion, chunking, embedding, vector indexing, retrieval, prompt engineering, LLM answer generation, and Gradio UI. Development tests confirmed useful answers for training, performance management, and workplace discrimination, plus a safe missing-information response for exact annual leave days.

**Limitations.** (1) `openrouter/free` can route to different models and response quality varies; (2) vector similarity is not proof of factual relevance; (3) some chunks cut across sentences or sections; (4) the performance query uses a narrow, hard-coded expansion; (5) the chatbot does not yet use conversation history; (6) it retrieves from the provided 2012 policy, not necessarily current HR policy, and cannot answer questions requiring documents outside that PDF.

**Potential improvements.** More systematic retrieval evaluation, metadata-aware source verification, keyword/semantic hybrid retrieval, a stable chosen OpenRouter model, and persistent hosting.

**Submission:** In Google Colab select **File → Download → Download .ipynb**. Never include a plaintext API key. Submit the notebook according to the assessment's LMS instructions, and supply the source PDF separately if the submission portal allows or requires it.